In [37]:
import pyarrow.parquet as pq
import pandas as pd
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import math
from collections import defaultdict

# INITS

In [178]:
def init_baseline_model(behaviors, history):
    """
    init_predict(dataset) dataset is from history data
    this is the learning phase

    my code needs history data to learn
    """
    # We want information about the articles that were clicked
    article_ids_scroll_percentages = {}
    article_ids_read_times = {}
    
    def build_scroll_percentage_and_read_time(row, article_ids_scroll_percentages, article_ids_read_times):
        for article_id, scroll_percentage_fixed, read_time_fixed in zip(row['article_id_fixed'], row['scroll_percentage_fixed'], row['read_time_fixed']):
            # print(article_ids_scroll_percentages.get(article_id, []))
            if article_id not in article_ids_scroll_percentages:
                article_ids_scroll_percentages[article_id] = []
            if article_id not in article_ids_read_times:
                article_ids_read_times[article_id] = []

            article_ids_read_times[article_id].append(read_time_fixed)
            article_ids_scroll_percentages[article_id].append(scroll_percentage_fixed)

            # article_ids_scroll_percentages[article_id] = article_ids_scroll_percentages.get(article_id, []).append(row['scroll_percentage_fixed'])
            # article_ids_read_times[article_id] = article_ids_read_times.get(article_id, []).append(row['read_time_fixed'])      

    history.apply(build_scroll_percentage_and_read_time, axis=1, args=(article_ids_scroll_percentages, article_ids_read_times))
    
    # we compute both clicked_sum and inview_sum for each article_id
    article_ids_clicked_sum = {}
    for i in behaviors['article_ids_clicked']:
        for j in i:
            article_ids_clicked_sum[j] = article_ids_clicked_sum.get(j, 0) + 1

    article_ids_inview_sum = {}
    for i in behaviors['article_ids_inview']:
        for j in i:
            article_ids_inview_sum[j] = article_ids_inview_sum.get(j, 0) + 1

    # this way we can compute the click rate for each article_id and sort them
    efficiency = pd.DataFrame({'clicked': article_ids_clicked_sum, 'inview': article_ids_inview_sum})
    efficiency['article_id'] = efficiency.index
    efficiency.reset_index(drop=True, inplace=True)
    efficiency['clicked'] = efficiency['clicked'].fillna(0)

    efficiency['click_rate'] = efficiency['clicked'] / efficiency['inview']
    
    efficiency = efficiency.sort_values(by='click_rate', ascending=False)

    # we can also comute the average scroll percentage and read time for each article_id
    article_ids_scroll_percentage_mean = {}
    for article_id, scroll_percentages in article_ids_scroll_percentages.items():
        article_ids_scroll_percentage_mean[article_id] = np.mean(scroll_percentages)
    article_ids_read_time_mean = {}
    for article_id, read_times in article_ids_read_times.items():
        article_ids_read_time_mean[article_id] = np.mean(read_times)

    interest = pd.DataFrame({'scroll_percentage': article_ids_scroll_percentage_mean, 'read_time': article_ids_read_time_mean})
    interest['article_id'] = interest.index
    interest.reset_index(drop=True, inplace=True)
    
    efficiency = pd.merge(efficiency, interest, on='article_id', how='outer')
    return efficiency

In [179]:
def init_content_model(articles, history):
    """
    dataset is from history data
    this is the learning phase
    """
    articles['text'] = (
        articles['title'].fillna('') + ' ' +
        articles['subtitle'].fillna('') + ' ' +
        articles['body'].fillna('')
    )

    article_text = articles['text']

    article_to_index = pd.Series(articles.index, index=articles['article_id']).drop_duplicates()

    article_titles = articles['title']

    danish_stopwords = [
        "og", "i", "det", "er", "som", "på", "de", "en", "til", "med", "at", "for", "der", "af", "han"
    ]

    tfidf = TfidfVectorizer(stop_words=danish_stopwords)

    article_matrix = tfidf.fit_transform(article_text)
    
    """
    Crée un profil moyen pour chaque utilisateur basé sur les articles qu'il a vus.
    Retourne un dictionnaire : {user_id: user_profile_vector}
    """
    user_profiles = {}

    for user_id in history['user_id'].unique():

        user_articles = history[history['user_id'] == user_id]['article_id_fixed'].explode().unique()
        
        indices = [article_to_index[aid] for aid in user_articles if aid in article_to_index.index]
        
        if not indices:
            continue  # Si aucun article connu, on saute
        
        user_profile = article_matrix[indices].mean(axis=0)
        user_profiles[user_id] = np.asarray(user_profile)

    return user_profiles, article_matrix, article_to_index


In [180]:
def init_collab_model(behaviors, history):
    user_items = defaultdict(set)

    # behaviors dataset: each row has one article_id
    for _, row in behaviors.iterrows():
        u = row['user_id']
        art = row['article_id']
        if pd.notna(art):        
            user_items[u].add(str(art))

    # history dataset: each row has article_id_fixed, an iterable
    for _, row in history.iterrows():
        u = row['user_id']
        arts = row['article_id_fixed']
        # skip if not iterable or empty
        try:
            iterator = iter(arts)
        except TypeError:
            continue
        for art in arts:
            user_items[u].add(str(art))
            
    # number of all users
    item_count = defaultdict(int)      
    # number of the intersection of all users and articles
    co_count   = defaultdict(lambda: defaultdict(int))  # |U_i ∩ U_j|

    for u, arts in user_items.items():
        arts = list(arts)
        # Count each item's total users
        for art in arts:
            item_count[art] += 1
        # For every pair (i,j) read by this user, increment co-occurrence both ways
        for idx in range(len(arts)):
            for jdx in range(idx+1, len(arts)):
                i, j = arts[idx], arts[jdx]
                co_count[i][j] += 1
                co_count[j][i] += 1

    return item_count, co_count

# SCORES

In [181]:
def baseline_model_score(article_id, efficiency=efficiency):
    """
    needs efficiency
    """
    if article_id not in efficiency['article_id'].values:
        # print(f"Article ID {article_id} not found in efficiency data.")
        return 0.0
    else:
        return efficiency[efficiency['article_id'] == article_id]['click_rate'].values[0]

In [182]:
def content_model_score(article_id, user_id, user_profiles=user_profiles, article_matrix=article_matrix, article_to_index=article_to_index):
    """
    Calcule un score de similarité entre un user_id et un article_id
    """
    if user_id not in user_profiles:
        # print(f"User ID {user_id} not found in user_profiles.")
        return 0.0  # Pas de profil pour cet utilisateur

    user_profile = user_profiles[user_id]

    if article_id not in article_to_index.index:
        # print(f"Article ID {article_id} not found in article_to_index.")
        return 0.0  # Pas d'article connu

    article_idx = article_to_index[article_id]

    article_vector = article_matrix[article_idx]

    # Calcul de la similarité cosine
    sim_score = cosine_similarity(user_profile.reshape(1, -1), article_vector.reshape(1, -1)).flatten()[0]

    return sim_score


In [183]:
def collab_model_score(article_id, user_id, history=history, co_count=co_count, item_count=item_count):
    user_id = int(user_id)
    
    # Articles déjà lus par l'utilisateur
    user_articles = history[history['user_id'] == user_id]['article_id_fixed'].explode().unique()

    # Si l'utilisateur n'a lu aucun article, retourner 0
    # Si l'utilisateur n'a lu aucun article, retourner un score de 0
    if len(user_articles) == 0:
        return 0.0

    # Convertir l'article_id en chaîne de caractères pour correspondre aux clés de co_count
    i = str(article_id)
    
    # Si l'article_id n'est pas dans co_count, retourner un score de 0
    if i not in co_count:
        return 0.0

    score = 0.0
    # Calculer la norme de l'article cible
    norm_i = math.sqrt(item_count[i])

    # Parcourir les articles lus par l'utilisateur
    for user_article in user_articles:
        user_article = str(user_article)
        # Vérifier si l'article lu par l'utilisateur est dans co_count pour l'article cible
        if user_article in co_count[i]:
            cij = co_count[i][user_article]  # Co-occurrence entre les deux articles
            norm_j = math.sqrt(item_count[user_article])  # Norme de l'article lu
            # Ajouter la contribution au score en utilisant la similarité
            score += cij / (norm_i * norm_j)

    # Retourner le score final
    return score


In [184]:
def hybrid_score(user_id, article_id):
    baseline_score = baseline_model_score(article_id)
    content_score = content_model_score(article_id, user_id)
    collab_score = collab_model_score(article_id, user_id)

    w1, w2, w3 = 0.2, 0.4, 0.4

    # normalize if needed
    # baseline_score = normalize(baseline_score)
    # content_score = normalize(content_score)
    # collab_score = normalize(collab_score)

    final_score = w1 * baseline_score + w2 * content_score + w3 * collab_score
    return final_score


# MAIN

In [186]:
articles_path = os.path.join('datas', 'ebnerd_demo', 'articles.parquet')
articles = pq.read_table(articles_path).to_pandas()

history_path = os.path.join('datas', 'ebnerd_demo', 'train', 'history.parquet')
history = pq.read_table(history_path).to_pandas()

behaviors_path = os.path.join('datas', 'ebnerd_demo', 'train', 'behaviors.parquet')
behaviors = pq.read_table(behaviors_path).to_pandas()

## INITS

In [163]:
efficiency = init_baseline_model(behaviors, history)

In [14]:
user_profiles, article_matrix, article_to_index = init_content_model(articles, history)

In [ ]:
item_count, co_count = init_collab_model(behaviors, history)

## INFERENCE

In [69]:
print(baseline_model_score(9749224, efficiency))

Article ID 9749224 not found in efficiency data.
0.0


In [67]:
print(content_model_score(9749224, 750497, user_profiles, article_matrix, article_to_index))

0.10744978972140608


In [68]:
print(collab_model_score(9749224, 750497, history, co_count, item_count))

39.10299870818089


In [72]:
hybrid_score(750497, 9749224)

Article ID 9749224 not found in efficiency data.


15.68417939916092

# PREDICT

In [187]:
def predict(row):
    """
    This is the prediction function
    takes the list of the articles_ids_inviews and returns the article_id to recommend
    """
    user_id = row['user_id']
    article_ids_inviews = row['article_ids_inview']
    best_article_id = None
    best_score = float('-inf')
    
    for article_id in article_ids_inviews:
        # we compute the score for each article_id
        score = hybrid_score(user_id, article_id)
        # print(f"Article ID: {article_id}, Score: {score}")
        
        # update the best article if the current score is higher
        if score > best_score:
            best_score = score
            best_article_id = article_id
    
    return best_article_id

# ACCURACY
We will use history to train the models then check the accuracy on behaviors with a window accuracy method.

In [188]:
data_start = behaviors['impression_time'].min()
data_end = behaviors['impression_time'].max()
data_start, data_end # the time range of the behaviors

(Timestamp('2023-05-18 07:00:03'), Timestamp('2023-05-25 06:59:52'))

In [189]:
def slide(data, window_size, slide_size):
    start = data_start
    end = start + window_size
    while end <= data_end:
        yield data[(data['impression_time'] >= start) & (data['impression_time'] < end)], start, end
        start += slide_size
        end = start + window_size

In [ ]:
for window, start, end in slide(behaviors, window_size=pd.Timedelta(days=3), slide_size=pd.Timedelta(days=1)):
    # we split the window into training and test set
    splitting_date = end - pd.Timedelta(days=1)
    # the first days are used for training and the last day for testing
    training = window[window['impression_time'] < splitting_date].copy()
    test = window[window['impression_time'] >= splitting_date].copy()

    # we train the models
    print(f"Training the model for the window {start} to {end}")
    print("Training the baseline model")
    efficiency = init_baseline_model(training, history)
    print("Training the content model")
    user_profiles, article_matrix, article_to_index = init_content_model(articles, history)
    print("Training the collaborative model")
    item_count, co_count = init_collab_model(training, history)

    print(f"Testing the model for the window {start} to {end}")
    # we test the model
    test.loc[:,'recommended_article_id'] = test.apply(predict, axis=1)
    # we compute the accuracy
    accuracy = (test['recommended_article_id'] == test['article_id']).mean()
    print(f"accuracy for the window {start} to {end} is {accuracy}")

Training the model for the window 2023-05-18 07:00:03 to 2023-05-21 07:00:03
Training the baseline model
Training the content model
Training the collaborative model
Testing the model for the window 2023-05-18 07:00:03 to 2023-05-21 07:00:03
accuracy for the window 2023-05-18 07:00:03 to 2023-05-21 07:00:03 is 0.008983890954151178
Training the model for the window 2023-05-19 07:00:03 to 2023-05-22 07:00:03
Training the baseline model
Training the content model
Training the collaborative model
Testing the model for the window 2023-05-19 07:00:03 to 2023-05-22 07:00:03
